In [7]:
import openeo
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from pathlib import Path
import numpy as np
import pandas as pd
from openeo.extra.job_management import MultiBackendJobManager
from openeo.processes import array_element, quantiles

import time

In [2]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [46]:
base_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/")

In [48]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [62]:
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]
BAD_SCL = [1, 3, 7, 8, 9, 10]
SPECTRAL = [b for b in BANDS if b != "SCL"]


def load_s2(spatial_extent, temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=BANDS,  # includes SCL
        max_cloud_cover=20,
    )
    return cube.resample_spatial(projection=32632, resolution=10, method="bilinear")


def cloud_mask(cube):
    scl = cube.band("SCL")
    m = scl == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (scl == v)
    return m


def load_masked(spatial_extent, temporal_extent):
    cube = load_s2(spatial_extent, temporal_extent)
    return cube.mask(cloud_mask(cube)).filter_bands(SPECTRAL)  # drops SCL after masking


cube_2016 = load_masked(spatial_extent_east, ("2016-07-01", "2016-10-01"))
cube_2017 = load_masked(spatial_extent_east, ("2017-07-01", "2017-10-01"))

cube = cube_2016.merge_cubes(cube_2017, overlap_resolver="min")

mosaic = cube.reduce_dimension(
    dimension="t",
    reducer=lambda x: array_element(quantiles(x, probabilities=[0.25]), index=0),
)

result = mosaic.save_result(format="GTiff")

In [63]:
job = result.create_job(title="S2_mosaic_east_2016_2017_test")
job.start_and_wait()
job.download_result(base_folder / "2016_2017_test.tif")

0:00:00 Job 'j-26101913392846ecb2afa8e1ecc0ed71': send 'start'
0:00:01 Job 'j-26101913392846ecb2afa8e1ecc0ed71': created (progress 0%)
0:00:06 Job 'j-26101913392846ecb2afa8e1ecc0ed71': queued (progress 0%)
0:00:13 Job 'j-26101913392846ecb2afa8e1ecc0ed71': queued (progress 0%)
0:00:21 Job 'j-26101913392846ecb2afa8e1ecc0ed71': queued (progress 0%)
0:00:31 Job 'j-26101913392846ecb2afa8e1ecc0ed71': queued (progress 0%)
0:00:43 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:00:58 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:01:17 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:01:41 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:02:11 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:02:48 Job 'j-26101913392846ecb2afa8e1ecc0ed71': running (progress N/A)
0:03:35 Job 'j-26101913392846ecb2afa8e1ecc0ed71': finished (progress 100%)


PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2016_2017_test.tif')

In [64]:
import time

BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
BAD_SCL = [1, 3, 7, 8, 9, 10]

years = list(range(2016, 2027))
year_pairs = [(y, y + 1) for y in years[:-1]]

def load_s2(spatial_extent, temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=BANDS + ["SCL"],
        max_cloud_cover=20,
    )
    return cube.resample_spatial(projection=32632, resolution=10, method="bilinear")


def cloud_mask(cube):
    scl = cube.band("SCL")
    m = scl == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (scl == v)
    return m


def create_s2_mosaic(spatial_extent, temporal_extents):
    cubes = [load_s2(spatial_extent, te) for te in temporal_extents]
    cube = cubes[0]
    for c in cubes[1:]:
        cube = cube.merge_cubes(c, overlap_resolver="max")

    mask = cloud_mask(cube)
    return (
        cube.mask(mask)
        .filter_bands(BANDS)
        .reduce_dimension(
            dimension="t",
            reducer=lambda x: array_element(quantiles(x, probabilities=[0.25]), index=0),
        )
    )


# ---- run ----
regions = {"west": spatial_extent_west, "east": spatial_extent_east}

for y1, y2 in year_pairs:
    temporal_extents = [
        [f"{y}-07-01T00:00:00Z", f"{y}-09-30T23:59:59Z"] for y in (y1, y2)
    ]

    results_folder = base_folder / f"{y1}_{y2}"
    results_folder.mkdir(parents=True, exist_ok=True)

    jobs = {}

    for name, extent in regions.items():
        out_file = results_folder / f"S2_mosaic_{name}_{y1}_{y2}.tiff"
        if out_file.exists():  # resume after interruption
            print(f"Skipping {out_file}, already downloaded")
            continue

        result = create_s2_mosaic(extent, temporal_extents).save_result(format="GTiff")
        job = result.create_job(title=f"S2_mosaic_{name}_{y1}_{y2}")
        job.start()
        jobs[name] = job
        print(f"{name} {y1}-{y2} started: {job.job_id}")
        time.sleep(10)

    for name, job in jobs.items():
        while job.status() not in ("finished", "error", "canceled"):
            time.sleep(30)

        if job.status() == "finished":
            out_file = results_folder / f"S2_mosaic_{name}_{y1}_{y2}.tiff"
            job.get_results().download_file(out_file)
            print(f"Downloaded {out_file}")
        else:
            print(f"Job {name} {y1}-{y2} ended with status {job.status()}")

west 2016-2017 started: j-26101914012041d4ab083f20cb904a9e
Skipping /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2016_2017/S2_mosaic_east_2016_2017.tiff, already downloaded
Downloaded /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2016_2017/S2_mosaic_west_2016_2017.tiff
west 2017-2018 started: j-2610191406514935ad4b8b4f393ccf4b
east 2017-2018 started: j-26101914070444fea008b11c119de202
Downloaded /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2017_2018/S2_mosaic_west_2017_2018.tiff
Downloaded /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2017_2018/S2_mosaic_east_2017_2018.tiff
west 2018-2019 started: j-26101914125942f89cfbc7e71f050bcc
east 2018-2019 started: j-2610191413124cb590883fbc4714825d
Downloaded /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2018_2019/S2_mosaic_west_2018_2019.tiff
Downloaded /mnt/CEPH_PROJECTS/provinz

In [67]:
import rasterio

indices_out = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/S2_indices")

for y1, y2 in year_pairs:
    temporal_extents = [
        [f"{y}-07-01T00:00:00Z", f"{y}-09-30T23:59:59Z"] for y in (y1, y2)
    ]
    
    in_folder = base_folder / f"{y1}_{y2}"
    out_folder = base_folder / f"{y1}_{y2}"

    print("Looking in:", in_folder)
    print("Folder exists:", in_folder.exists())

    for f in sorted(in_folder.glob("S2_mosaic*.tiff")):
        with rasterio.open(f) as src:
            print(f"File: {f.name}, CRS: {src.crs}, Size: {src.width}x{src.height}")

            green = src.read(2).astype(np.float32)
            red   = src.read(3).astype(np.float32)
            nir   = src.read(4).astype(np.float32)
            swir  = src.read(5).astype(np.float32)
            nodata = src.nodata
            profile = src.profile.copy()

        # Valid-pixel mask
        valid = (swir > 0) & (red >= 0) & (green >= 0) & (nir >= 0)
        if nodata is not None:
            valid &= (red != nodata) & (swir != nodata) & (green != nodata) & (nir != nodata)

        with np.errstate(divide="ignore", invalid="ignore"):
            red_swir_ratio = np.where(valid, red / swir, np.nan)
            ndsi = np.where(valid, (green - swir) / (green + swir), np.nan)

        # apply treshold to both 
        ndsi_thresh = ndsi > 0.6
        ratio_thresh = red_swir_ratio > 10

        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

        outputs = {
            "red_swir_ratio": red_swir_ratio,
            "ndsi": ndsi,
            "ndsi_0.6" : ndsi_thresh,
            "ratio_10": ratio_thresh
        }

        for name, arr in outputs.items():
            with rasterio.open(out_folder / f"{f.stem}_{name}.tiff", "w", **profile) as dst:
                dst.write(arr.astype(np.float32), 1)

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2016_2017
Folder exists: True
File: S2_mosaic_east_2016_2017.tiff, CRS: EPSG:32632, Size: 4294x2378


/tmp/ipykernel_3260193/1062981596.py:20: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  green = src.read(2).astype(np.float32)
/tmp/ipykernel_3260193/1062981596.py:21: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  red   = src.read(3).astype(np.float32)
/tmp/ipykernel_3260193/1062981596.py:22: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  nir   = src.read(4).astype(np.float32)
/tmp/ipykernel_3260193/1062981596.py:23: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if ne

File: S2_mosaic_west_2016_2017.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2017_2018
Folder exists: True
File: S2_mosaic_east_2017_2018.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2017_2018.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2018_2019
Folder exists: True
File: S2_mosaic_east_2018_2019.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2018_2019.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2019_2020
Folder exists: True
File: S2_mosaic_east_2019_2020.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2019_2020.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics_twoYears/2020_2021
Folder exists: True
File: S2_mosaic_east_2020_